##### make_timestamp()

- creates a **timestamp** from individual **date and time** components (**year, month, day, hour, minute, second**).
- `All inputs must be integers`.
- It is available in `Spark 3.0+`.

#### Syntax

     import pyspark.sql.functions as F

     # From individual components
     F.make_timestamp(years=<years>, months=<months>, days=<days>, hours=<hours>, mins=<mins>, secs=<secs>)

     # With timezone
     F.make_timestamp(years=<years>, months=<months>, days=<days>, hours=<hours>, mins=<mins>, secs=<secs>, timezone=<timezone>)

     # From date and time
     F.make_timestamp(date=<date>, time=<time>)

     # From date and time with timezone
     F.make_timestamp(date=<date>, time=<time>, timezone=<timezone>)

| Argument  |     Description                    |
|-----------|------------------------------------|
|  year     | An integer expression (from 1 to 9999). |
|  month    | An integer expression (from 1 to 12). |
|  day      | An integer expression (from 1 to 31). |
|  hour     | An integer expression (from 0 to 23). |
|  min      | An integer expression (from 0 to 59). |
|  sec      | A numeric expression (integer or fractional, from 0 to 60). The value can be either an integer like 13, or a fraction like 13.123.|
| timezone  | Optional. The time zone identifier. (e.g., 'UTC', 'CET', 'America/Los_Angeles'). |

**Returns:**
- A new column that contains a `timestamp`.

**Content:**
- `Make timestamp from years, months, days, hours, mins, secs and timezone`.
- `Make timestamp from years, months, days, hours, mins, and secs (without timezone)`.
- `Make timestamp from date and time (without timezone).`
- `Create a Timestamp from Separate Columns.`
- `Create a Timestamp from Literal Values.`
- `Create a Timestamp with Error Handling.`
- `Handling Invalid Values.`
- `Using Spark SQL`.

In [0]:
from pyspark.sql.functions import col, lit, make_timestamp

##### 1) make_timestamp() from `years, months, days, hours, mins, secs and timezone`.

In [0]:
# Sample data
data = [(2018, 2, 14, 10, 0, 45.887, 'CET')]

schema = ["year", "month", "day", "hour", "minute", "second", "tz"]

df_ts = spark.createDataFrame(data, schema)
display(df_ts)

year,month,day,hour,minute,second,tz
2018,2,14,10,0,45.887,CET


In [0]:
# Create the timestamp column
df_ts_final = df_ts.withColumn(
    "timestamp_col",
    make_timestamp(col("year"), col("month"), col("day"), col("hour"), col("minute"), col("second"), col("tz"))
)

display(df_ts_final)

year,month,day,hour,minute,second,tz,timestamp_col
2018,2,14,10,0,45.887,CET,2018-02-14T09:00:45.887Z


##### 2) make_timestamp() from years, months, days, hours, mins, and secs (without timezone)

In [0]:
# Sample data
data = [(2018, 2, 14, 10, 0, 45.887)]

schema = ["year", "month", "day", "hour", "minute", "second"]

df_ts_wo_tz = spark.createDataFrame(data, schema)
display(df_ts_wo_tz)

year,month,day,hour,minute,second
2018,2,14,10,0,45.887


In [0]:
# Create the timestamp column
df_ts_wo_tz_final = df_ts_wo_tz.withColumn(
    "timestamp_col",
    make_timestamp(col("year"), col("month"), col("day"), col("hour"), col("minute"), col("second"))
)

display(df_ts_wo_tz_final)

year,month,day,hour,minute,second,timestamp_col
2018,2,14,10,0,45.887,2018-02-14T10:00:45.887Z


In [0]:
# spark.conf.set("spark.sql.session.timeZone", "America/Los_Angeles")
# spark.conf.unset("spark.sql.session.timeZone")

##### 3) make_timestamp() from date and time (without timezone).

In [0]:
from pyspark.sql.functions import to_timestamp, concat, lit

# Create a DataFrame with a date, time, and timezone
df = spark.range(1).select(
    lit("2014-12-28").alias("date"),
    lit("06:30:45").alias("time")
)
display(df)

# Concatenate the date and time into a string
df = df.withColumn("datetime_str", concat(df.date, lit(" "), df.time))
# Create a timestamp from the datetime string with timezone
df_final = df.withColumn("timestamp", to_timestamp(df.datetime_str, "yyyy-MM-dd HH:mm:ss"))
# Display the result
display(df_final)

date,time
2014-12-28,06:30:45


date,time,datetime_str,timestamp
2014-12-28,06:30:45,2014-12-28 06:30:45,2014-12-28T06:30:45.000Z


##### 4) Create a Timestamp from Separate Columns.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Sample data
data = [
    (2019, 2, 13, 10, 30, 0, 'UTC'),
    (2018, 2, 14, 10, 0, 45.887, 'CET'),
    (2022, 12, 31, 14, 45, 20, 'UTC'),
    (2020, 10, 2, 18, 0, 0, 'America/Los_Angeles'),
    (2020, 10, 2, 18, 0, None, 'America/Los_Angeles')
]
schema = StructType([
    StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("day", IntegerType(), True),
    StructField("hour", IntegerType(), True),
    StructField("minute", IntegerType(), True),
    StructField("second", IntegerType(), True),
    StructField("tz", StringType(), True)
])

df_ts = spark.createDataFrame(data, schema)
display(df_ts)

year,month,day,hour,minute,second,tz
2019,2,13,10,30,0,UTC
2018,2,14,10,0,45,CET
2022,12,31,14,45,20,UTC
2020,10,2,18,0,0,America/Los_Angeles
2020,10,2,18,0,null,America/Los_Angeles


In [0]:
# Create the timestamp column
df_ts_final = df_ts.withColumn(
    "timestamp_col",
    F.make_timestamp(F.col("year"), F.col("month"), F.col("day"), F.col("hour"), F.col("minute"), F.col("second"), F.col("tz"))
)

display(df_ts_final)

year,month,day,hour,minute,second,tz,timestamp_col
2019,2,13,10,30,0,UTC,2019-02-13T10:30:00.000Z
2018,2,14,10,0,45,CET,2018-02-14T09:00:45.000Z
2022,12,31,14,45,20,UTC,2022-12-31T14:45:20.000Z
2020,10,2,18,0,0,America/Los_Angeles,2020-10-03T01:00:00.000Z
2020,10,2,18,0,null,America/Los_Angeles,null


##### 5) Create a Timestamp from Literal Values

In [0]:
from pyspark.sql.functions import lit, make_timestamp

# Create a DataFrame with a single row
df_lit = spark.createDataFrame([(1,)], ["id"])

# Create a timestamp column using the make_timestamp() function with literal values
df_lit = df_lit \
    .withColumn("new_year_midnight", make_timestamp(lit(2026), lit(1), lit(1), lit(0), lit(0), lit(0))) \
    .withColumn("new_year_midnight_null", make_timestamp(lit(2026), lit(1), lit(1), lit(0), lit(0), lit(None)))

display(df_lit)

id,new_year_midnight,new_year_midnight_null
1,2026-01-01T00:00:00.000Z,null


##### 6) Create a Timestamp with Error Handling

In [0]:
# Create a DataFrame with separate year, month, day, hour, minute, and second columns
data = [(2022, 1, 1, 10, 30, 0),
        (2023, 1, 1, 14, 45, 0),
        (2022, 13, 31, 23, 59, 59)]
        
columns = ["year_col", "month_col", "day_col", "hour_col", "minute_col", "second_col"]

df_error = spark.createDataFrame(data, columns)
display(df_error)

year_col,month_col,day_col,hour_col,minute_col,second_col
2022,1,1,10,30,0
2023,1,1,14,45,0
2022,13,31,23,59,59


In [0]:
from pyspark.sql.functions import col, when

# Create a timestamp column using the make_timestamp() function with error handling
df_error_final_01 = df_error \
    .withColumn("timestamp_col", 
        make_timestamp(col("year_col"), col("month_col"), col("day_col"), col("hour_col"), col("minute_col"), col("second_col")))

# Display the result
display(df_error_final_01)

---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-7047824761388816>, line 9
      4 df_error_final_01 = df_error \
      5     .withColumn("timestamp_col", 
      6         make_timestamp(col("year_col"), col("month_col"), col("day_col"), col("hour_col"), col("minute_col"), col("second_col")))
      8 # Display the result
----> 9 display(df_error_final_01)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
    135     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:97, in Display.display_connect_table(self, df, **kwargs)
     94     self.cf_helper.display_streaming_dataframe(df, conf

**Method 01**

In [0]:
# spark.conf.set("spark.sql.ansi.enabled", True)
# spark.conf.set("spark.sql.ansi.enabled", False)

In [0]:
from pyspark.sql.functions import col, when

# Create a timestamp column using the make_timestamp() function with error handling
df_error_final_01 = df_error \
    .withColumn("timestamp_col", 
        make_timestamp(col("year_col"), col("month_col"), col("day_col"), col("hour_col"), col("minute_col"), col("second_col")))

# Display the result
display(df_error_final_01)

year_col,month_col,day_col,hour_col,minute_col,second_col,timestamp_col
2022,1,1,10,30,0,2022-01-01T10:30:00.000Z
2023,1,1,14,45,0,2023-01-01T14:45:00.000Z
2022,13,31,23,59,59,null


**Method 02**

In [0]:
from pyspark.sql.functions import col, when

# Create a timestamp column using the make_timestamp() function with error handling
df_error_final = df_error \
    .withColumn("timestamp_col", when(
        (col("month_col") >= 1) & (col("month_col") <= 12),
        make_timestamp(col("year_col"), col("month_col"), col("day_col"), col("hour_col"), col("minute_col"), col("second_col"))) \
            .otherwise(lit(None)))

# Display the result
display(df_error_final)

year_col,month_col,day_col,hour_col,minute_col,second_col,timestamp_col
2022,1,1,10,30,0,2022-01-01T10:30:00.000Z
2023,1,1,14,45,0,2023-01-01T14:45:00.000Z
2022,13,31,23,59,59,null


##### 7) Handling Invalid Values

In [0]:
data = [(2026, 2, 30, 10, 0, 0)]
df_inval = spark.createDataFrame(data, ["year", "month", "day", "hour", "minute", "second"])

df_inval.select(
    make_timestamp("year", "month", "day", "hour", "minute", "second")
).display()

---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-6743578884386988>, line 6
      1 data = [(2026, 2, 30, 10, 0, 0)]
      2 df_inval = spark.createDataFrame(data, ["year", "month", "day", "hour", "minute", "second"])
      4 df_inval.select(
      5     make_timestamp("year", "month", "day", "hour", "minute", "second")
----> 6 ).display()

File /databricks/python_shell/lib/dbruntime/monkey_patches.py:74, in apply_dataframe_display_patch.<locals>.df_display(df, *args, **kwargs)
     70 def df_display(df, *args, **kwargs):
     71     """
     72     df.display() is an alias for display(df). Run help(display) for more information.
     73     """
---> 74     display(df, *args, **kwargs)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input,

In [0]:
data = [(2026, 2, 30, 10, 0, 0)]
df_inval = spark.createDataFrame(data, ["year", "month", "day", "hour", "minute", "second"])

df_inval.select(
    make_timestamp("year", "month", "day", "hour", "minute", "second").alias("invalid_col")
).display()

invalid_col
null


In [0]:
%sql
SELECT make_timestamp(2019, 13, 1, 10, 11, 12, 'PST');

---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-6743578884386996>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "SELECT make_timestamp(2019, 13, 1, 10, 11, 12, 'PST');\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:192, in SqlMagic.sql(self, line, cell)
    186 except BaseException as e:
    187     self.driver

In [0]:
%sql
SELECT make_timestamp(2019, 13, 1, 10, 11, 12, 'PST') AS invalid_col;

invalid_col
null


##### 8) Using Spark SQL

In [0]:
df_ts.createOrReplaceTempView("time_table")

spark.sql("""
SELECT
    year,
    month,
    day,
    hour,
    minute,
    second,
    make_timestamp(year, month, day, hour, minute, second) AS full_ts
FROM time_table
""").display()

year,month,day,hour,minute,second,full_ts
2019,2,13,10,30,0,2019-02-13T10:30:00.000Z
2018,2,14,10,0,45,2018-02-14T10:00:45.000Z
2022,12,31,14,45,20,2022-12-31T14:45:20.000Z
2020,10,2,18,0,0,2020-10-02T18:00:00.000Z
2020,10,2,18,0,null,null


In [0]:
%sql
SELECT make_timestamp(NULL, 7, 22, 15, 30, 0) as Null_Value;

Null_Value
null


In [0]:
%sql
SELECT make_timestamp(2019, 6, 30, 23, 59, 60)
UNION ALL
SELECT make_timestamp(2019, 6, 30, 23, 59, 60, 'UTC')
UNION ALL
SELECT make_timestamp(2019, 6, 30, 22, 19, 20, 'America/Los_Angeles')
UNION ALL
SELECT make_timestamp(2019, 6, 30, 23, 59, 60, 'CET')
UNION ALL
SELECT make_timestamp(2014, 12, 28, 6, 30, 45.887)
UNION ALL
SELECT make_timestamp(2014, 12, 28, 6, 30, 45.887, 'CET')
UNION ALL
SELECT make_timestamp(2019, 11, 1, 10, 11, 12, 'PST');

"make_timestamp(2019,6,30,23,59,60)"
2019-07-01T00:00:00.000Z
2019-07-01T00:00:00.000Z
2019-07-01T05:19:20.000Z
2019-06-30T22:00:00.000Z
2014-12-28T06:30:45.887Z
2014-12-28T05:30:45.887Z
2019-11-01T17:11:12.000Z
